# 19. Testing fakes, mocks, and invariants

**Level:** Intermediate to advanced

**Learning outcome:** Verify outcomes and required interactions using fake transports, autospec, injected clocks, and negative tests.

**Prerequisites:** 03-04 and 15: invariant boundaries and explicit errors.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](18_Desired_State_Idempotency_and_Rollback.ipynb) | [Next notebook](20_Serialization_Schemas_Enums_and_Snapshots.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Intermediate to advanced

**Learning goal:** Use fakes, mocks, injected time, rejection checks, and behavioral invariants.

**Test the promise, not the spelling of the implementation.** A good test demonstrates that a
change is rejected, a result is correct, or a dependency is called with required arguments.
Tests that merely check a constant or repeat the implementation's exact expression can miss
the real requirement. Use small known examples with independently understandable answers.

**A fake** is a usable simplified implementation, such as an in-memory transport with a call
log. **A mock** records interactions and can assert required calls. Autospeccing constrains a
mock to a known API, reducing accidental acceptance of misspelled methods. Neither replaces
integration testing against a reviewed lab environment.

**Inject time.** Expiry, retries, and cache tests should not sleep for real seconds. Pass a
clock callable and advance a fake clock deliberately. This makes boundary conditions such as
exact expiry deterministic and fast.

**Negative tests** must prove rejection happened. In a try/except test, add an else branch
that fails if the expected exception did not occur. Check state after the failure so an
invalid update cannot partially mutate the object. In unittest use assertRaises for the same
intent. Catch a specific expected exception, not all errors.

**Property-style thinking** asks what should always hold: applying intent twice should produce
no second change; snapshots should not mutate when the source changes; normalization should
be stable on repeated application. These bounded examples use loops and assertions, not an
external property-testing dependency. Broader generated tests can complement, not replace,
explicit edge-case examples.

**Pause and explain:** Name one interaction worth mocking and one outcome better checked through a real fake.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 066: Test a fake's interaction log

**Core concept:** A deterministic fake can verify both results and required calls.

**Network-engineering purpose:** A status reader should issue exactly one read command for the selected interface.

### Read the implementation in this order
1. Record commands in a fake.
2. Inject it into a reader.
3. Run one read.
4. Assert exact command order and the returned response.

**Common mistake / boundary:** A fake that behaves unlike the real adapter can give false confidence.

**Practice after the ten worked scenarios:** Add an adapter contract test suite that can be reused for fake and lab implementations.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0651 - Test a fake's interaction log: branch scenario

**Scenario 01/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet1/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [1]:
# Example 0651: Test a fake's interaction log - branch
case = {'interface': 'Ethernet1/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet1/1']


### Example 0652 - Test a fake's interaction log: campus scenario

**Scenario 02/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet2/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [2]:
# Example 0652: Test a fake's interaction log - campus
case = {'interface': 'Ethernet2/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet2/1']


### Example 0653 - Test a fake's interaction log: dc scenario

**Scenario 03/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet3/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [3]:
# Example 0653: Test a fake's interaction log - dc
case = {'interface': 'Ethernet3/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet3/1']


### Example 0654 - Test a fake's interaction log: wan scenario

**Scenario 04/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet4/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [4]:
# Example 0654: Test a fake's interaction log - wan
case = {'interface': 'Ethernet4/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet4/1']


### Example 0655 - Test a fake's interaction log: lab scenario

**Scenario 05/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet5/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [5]:
# Example 0655: Test a fake's interaction log - lab
case = {'interface': 'Ethernet5/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet5/1']


### Example 0656 - Test a fake's interaction log: wireless scenario

**Scenario 06/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet6/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [6]:
# Example 0656: Test a fake's interaction log - wireless
case = {'interface': 'Ethernet6/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet6/1']


### Example 0657 - Test a fake's interaction log: remote scenario

**Scenario 07/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet7/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [7]:
# Example 0657: Test a fake's interaction log - remote
case = {'interface': 'Ethernet7/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet7/1']


### Example 0658 - Test a fake's interaction log: core scenario

**Scenario 08/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet8/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [8]:
# Example 0658: Test a fake's interaction log - core
case = {'interface': 'Ethernet8/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet8/1']


### Example 0659 - Test a fake's interaction log: mgmt scenario

**Scenario 09/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet9/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [9]:
# Example 0659: Test a fake's interaction log - mgmt
case = {'interface': 'Ethernet9/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet9/1']


### Example 0660 - Test a fake's interaction log: backup scenario

**Scenario 10/10; pattern 066; Intermediate to advanced.** This is a self-contained offline exercise. A status reader should issue exactly one read command for the selected interface.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |

**Logic to trace:** Record commands in a fake. Assert exact command order and the returned response. Follow the class-specific steps under Pattern 066 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['read Ethernet10/1']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A fake that behaves unlike the real adapter can give false confidence.

In [10]:
# Example 0660: Test a fake's interaction log - backup
case = {'interface': 'Ethernet10/1'}

class Fake:
    def __init__(self):
        self.commands = []

    def send(self, command):
        self.commands.append(command)
        return "sample"

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, name):
        return self.transport.send("read " + name)

fake = Fake()
assert Reader(fake).read(case["interface"]) == "sample"
assert fake.commands == ["read " + case["interface"]]
print(fake.commands)


['read Ethernet10/1']


## Pattern 067: Autospecced mock for a collaborator

**Core concept:** An autospecced mock checks interaction against a real API shape.

**Network-engineering purpose:** A transport call can be verified without enabling actual networking.

### Read the implementation in this order
1. Define the intended Transport API.
2. Create an instance mock using create_autospec.
3. Set its deterministic return value.
4. Assert the exact call and returned result.

**Common mistake / boundary:** Interaction checks alone do not prove that a parser or domain policy is correct.

**Practice after the ten worked scenarios:** Make a service accept this mock and assert it calls send once rather than testing the mock directly.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0661 - Autospecced mock for a collaborator: branch scenario

**Scenario 01/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [11]:
# Example 0661: Autospecced mock for a collaborator - branch
case = {'interface': 'Ethernet1/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0662 - Autospecced mock for a collaborator: campus scenario

**Scenario 02/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [12]:
# Example 0662: Autospecced mock for a collaborator - campus
case = {'interface': 'Ethernet2/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0663 - Autospecced mock for a collaborator: dc scenario

**Scenario 03/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [13]:
# Example 0663: Autospecced mock for a collaborator - dc
case = {'interface': 'Ethernet3/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0664 - Autospecced mock for a collaborator: wan scenario

**Scenario 04/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [14]:
# Example 0664: Autospecced mock for a collaborator - wan
case = {'interface': 'Ethernet4/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0665 - Autospecced mock for a collaborator: lab scenario

**Scenario 05/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [15]:
# Example 0665: Autospecced mock for a collaborator - lab
case = {'interface': 'Ethernet5/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0666 - Autospecced mock for a collaborator: wireless scenario

**Scenario 06/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [16]:
# Example 0666: Autospecced mock for a collaborator - wireless
case = {'interface': 'Ethernet6/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0667 - Autospecced mock for a collaborator: remote scenario

**Scenario 07/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [17]:
# Example 0667: Autospecced mock for a collaborator - remote
case = {'interface': 'Ethernet7/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0668 - Autospecced mock for a collaborator: core scenario

**Scenario 08/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [18]:
# Example 0668: Autospecced mock for a collaborator - core
case = {'interface': 'Ethernet8/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0669 - Autospecced mock for a collaborator: mgmt scenario

**Scenario 09/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [19]:
# Example 0669: Autospecced mock for a collaborator - mgmt
case = {'interface': 'Ethernet9/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


### Example 0670 - Autospecced mock for a collaborator: backup scenario

**Scenario 10/10; pattern 067; Intermediate to advanced.** This is a self-contained offline exercise. A transport call can be verified without enabling actual networking.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |

**Logic to trace:** Define the intended Transport API. Assert the exact call and returned result. Follow the class-specific steps under Pattern 067 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
expected transport interaction verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Interaction checks alone do not prove that a parser or domain policy is correct.

In [20]:
# Example 0670: Autospecced mock for a collaborator - backup
case = {'interface': 'Ethernet10/1'}

from unittest.mock import create_autospec

class Transport:
    def send(self, command: str) -> str:
        raise NotImplementedError

class Reader:
    def __init__(self, transport):
        self.transport = transport

    def read(self, interface):
        return self.transport.send(f"read {interface}")

transport = create_autospec(Transport, instance=True, spec_set=True)
transport.send.return_value = "up"
result = Reader(transport).read(case["interface"])
transport.send.assert_called_once_with(f"read {case['interface']}")
assert result == "up"
print("expected transport interaction verified")


expected transport interaction verified


## Pattern 068: Injected clock tests exact expiry

**Core concept:** Time-based state can be deterministic when time is a dependency.

**Network-engineering purpose:** An observation cache should reject a value at its documented expiry boundary.

### Read the implementation in this order
1. Use a callable clock with mutable test time.
2. Store an absolute expiry.
3. Read successfully before expiry.
4. Advance to exact expiry and verify a stale-data error.

**Common mistake / boundary:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

**Practice after the ten worked scenarios:** Test one instant just before expiry and one just after without any sleeps.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0671 - Injected clock tests exact expiry: branch scenario

**Scenario 01/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |
| `ttl` | `5` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [21]:
# Example 0671: Injected clock tests exact expiry - branch
case = {'hostname': 'branch-edge-01', 'ttl': 5}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0672 - Injected clock tests exact expiry: campus scenario

**Scenario 02/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |
| `ttl` | `10` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [22]:
# Example 0672: Injected clock tests exact expiry - campus
case = {'hostname': 'campus-access-02', 'ttl': 10}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0673 - Injected clock tests exact expiry: dc scenario

**Scenario 03/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |
| `ttl` | `15` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [23]:
# Example 0673: Injected clock tests exact expiry - dc
case = {'hostname': 'dc-leaf-03', 'ttl': 15}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0674 - Injected clock tests exact expiry: wan scenario

**Scenario 04/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |
| `ttl` | `20` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [24]:
# Example 0674: Injected clock tests exact expiry - wan
case = {'hostname': 'wan-border-04', 'ttl': 20}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0675 - Injected clock tests exact expiry: lab scenario

**Scenario 05/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |
| `ttl` | `25` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [25]:
# Example 0675: Injected clock tests exact expiry - lab
case = {'hostname': 'lab-router-05', 'ttl': 25}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0676 - Injected clock tests exact expiry: wireless scenario

**Scenario 06/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |
| `ttl` | `30` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [26]:
# Example 0676: Injected clock tests exact expiry - wireless
case = {'hostname': 'wireless-dist-06', 'ttl': 30}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0677 - Injected clock tests exact expiry: remote scenario

**Scenario 07/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |
| `ttl` | `35` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [27]:
# Example 0677: Injected clock tests exact expiry - remote
case = {'hostname': 'remote-gateway-07', 'ttl': 35}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected

### Example 0678 - Injected clock tests exact expiry: core scenario

**Scenario 08/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |
| `ttl` | `40` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [28]:
# Example 0678: Injected clock tests exact expiry - core
case = {'hostname': 'core-router-08', 'ttl': 40}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected

### Example 0679 - Injected clock tests exact expiry: mgmt scenario

**Scenario 09/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |
| `ttl` | `45` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [29]:
# Example 0679: Injected clock tests exact expiry - mgmt
case = {'hostname': 'mgmt-switch-09', 'ttl': 45}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


### Example 0680 - Injected clock tests exact expiry: backup scenario

**Scenario 10/10; pattern 068; Intermediate to advanced.** This is a self-contained offline exercise. An observation cache should reject a value at its documented expiry boundary.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |
| `ttl` | `50` |

**Logic to trace:** Use a callable clock with mutable test time. Advance to exact expiry and verify a stale-data error. Follow the class-specific steps under Pattern 068 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
exact expiry rejected
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Use a monotonic clock for elapsed-time logic in a real process; wall clocks can jump.

In [30]:
# Example 0680: Injected clock tests exact expiry - backup
case = {'hostname': 'backup-edge-10', 'ttl': 50}

class Clock:
    def __init__(self):
        self.now = 0

    def __call__(self):
        return self.now

class Cache:
    def __init__(self, clock, ttl, value):
        self.clock, self.expires, self.value = clock, clock() + ttl, value

    def get(self):
        if self.clock() >= self.expires:
            raise LookupError("observation expired")
        return self.value

clock = Clock()
cache = Cache(clock, case["ttl"], case["hostname"])
assert cache.get() == case["hostname"]
clock.now = case["ttl"]
try:
    cache.get()
except LookupError:
    print("exact expiry rejected")
else:
    raise AssertionError("stale observation returned")


exact expiry rejected


## Pattern 069: Unit test a rejected state transition

**Core concept:** A negative test checks both the exception and unchanged state.

**Network-engineering purpose:** A bad VLAN proposal must not overwrite the previous valid value.

### Read the implementation in this order
1. Define explicit runtime validation.
2. Use unittest.TestCase for a focused test.
3. Assert ValueError for the bad update.
4. Verify the existing VLAN is unchanged.

**Common mistake / boundary:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

**Practice after the ten worked scenarios:** Add subTest cases for 0, 4095, True, and a string.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0681 - Unit test a rejected state transition: branch scenario

**Scenario 01/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [31]:
# Example 0681: Unit test a rejected state transition - branch
case = {'vlan': 1}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0682 - Unit test a rejected state transition: campus scenario

**Scenario 02/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `10` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [32]:
# Example 0682: Unit test a rejected state transition - campus
case = {'vlan': 10}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0683 - Unit test a rejected state transition: dc scenario

**Scenario 03/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `100` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [33]:
# Example 0683: Unit test a rejected state transition - dc
case = {'vlan': 100}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0684 - Unit test a rejected state transition: wan scenario

**Scenario 04/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `200` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [34]:
# Example 0684: Unit test a rejected state transition - wan
case = {'vlan': 200}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0685 - Unit test a rejected state transition: lab scenario

**Scenario 05/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `999` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [35]:
# Example 0685: Unit test a rejected state transition - lab
case = {'vlan': 999}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0686 - Unit test a rejected state transition: wireless scenario

**Scenario 06/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `1000` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [36]:
# Example 0686: Unit test a rejected state transition - wireless
case = {'vlan': 1000}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0687 - Unit test a rejected state transition: remote scenario

**Scenario 07/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `2000` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [37]:
# Example 0687: Unit test a rejected state transition - remote
case = {'vlan': 2000}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0688 - Unit test a rejected state transition: core scenario

**Scenario 08/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `3000` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [38]:
# Example 0688: Unit test a rejected state transition - core
case = {'vlan': 3000}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0689 - Unit test a rejected state transition: mgmt scenario

**Scenario 09/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4000` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [39]:
# Example 0689: Unit test a rejected state transition - mgmt
case = {'vlan': 4000}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


### Example 0690 - Unit test a rejected state transition: backup scenario

**Scenario 10/10; pattern 069; Intermediate to advanced.** This is a self-contained offline exercise. A bad VLAN proposal must not overwrite the previous valid value.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `vlan` | `4093` |

**Logic to trace:** Define explicit runtime validation. Verify the existing VLAN is unchanged. Follow the class-specific steps under Pattern 069 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
rejection and unchanged state verified
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not test only that some exception occurred; a programming error might satisfy that weak test.

In [40]:
# Example 0690: Unit test a rejected state transition - backup
case = {'vlan': 4093}

import unittest

class AccessPort:
    def __init__(self, vlan):
        self.set_vlan(vlan)

    def set_vlan(self, vlan):
        if type(vlan) is not int or not 1 <= vlan <= 4094:
            raise ValueError("invalid VLAN")
        self.vlan = vlan

class PortTests(unittest.TestCase):
    def runTest(self):
        port = AccessPort(case["vlan"])
        with self.assertRaisesRegex(ValueError, "invalid VLAN"):
            port.set_vlan(0)
        self.assertEqual(port.vlan, case["vlan"])

result = unittest.TestResult()
PortTests().run(result)
assert result.wasSuccessful(), (result.errors, result.failures)
assert result.testsRun == 1
print("rejection and unchanged state verified")


rejection and unchanged state verified


## Pattern 070: Check an idempotency invariant across values

**Core concept:** Behavioral properties can be checked over a bounded set of meaningful inputs.

**Network-engineering purpose:** A reconciliation helper should do no extra work when the same intent is applied again.

### Read the implementation in this order
1. Create a reconciler with a write counter.
2. Try normal and boundary MTU values.
3. Apply each value once, then again.
4. Assert the second call leaves the write count unchanged.

**Common mistake / boundary:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

**Practice after the ten worked scenarios:** Add a property about final state and a rejection property for out-of-policy values.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0691 - Check an idempotency invariant across values: branch scenario

**Scenario 01/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [41]:
# Example 0691: Check an idempotency invariant across values - branch
case = {'mtu': 1500}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


### Example 0692 - Check an idempotency invariant across values: campus scenario

**Scenario 02/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [42]:
# Example 0692: Check an idempotency invariant across values - campus
case = {'mtu': 9000}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs

### Example 0693 - Check an idempotency invariant across values: dc scenario

**Scenario 03/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [43]:
# Example 0693: Check an idempotency invariant across values - dc
case = {'mtu': 9216}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs

### Example 0694 - Check an idempotency invariant across values: wan scenario

**Scenario 04/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [44]:
# Example 0694: Check an idempotency invariant across values - wan
case = {'mtu': 576}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


### Example 0695 - Check an idempotency invariant across values: lab scenario

**Scenario 05/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [45]:
# Example 0695: Check an idempotency invariant across values - lab
case = {'mtu': 1600}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


### Example 0696 - Check an idempotency invariant across values: wireless scenario

**Scenario 06/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [46]:
# Example 0696: Check an idempotency invariant across values - wireless
case = {'mtu': 1500}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


### Example 0697 - Check an idempotency invariant across values: remote scenario

**Scenario 07/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [47]:
# Example 0697: Check an idempotency invariant across values - remote
case = {'mtu': 2000}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


### Example 0698 - Check an idempotency invariant across values: core scenario

**Scenario 08/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [48]:
# Example 0698: Check an idempotency invariant across values - core
case = {'mtu': 4470}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs

### Example 0699 - Check an idempotency invariant across values: mgmt scenario

**Scenario 09/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [49]:
# Example 0699: Check an idempotency invariant across values - mgmt
case = {'mtu': 9100}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


### Example 0700 - Check an idempotency invariant across values: backup scenario

**Scenario 10/10; pattern 070; Intermediate to advanced.** This is a self-contained offline exercise. A reconciliation helper should do no extra work when the same intent is applied again.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |

**Logic to trace:** Create a reconciler with a write counter. Assert the second call leaves the write count unchanged. Follow the class-specific steps under Pattern 070 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
idempotency holds for normal and boundary inputs
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A bounded loop is useful evidence, not a mathematical proof over every possible input.

In [50]:
# Example 0700: Check an idempotency invariant across values - backup
case = {'mtu': 8192}

class Reconciler:
    def __init__(self):
        self.value, self.writes = None, 0

    def ensure(self, value):
        if self.value != value:
            self.value = value
            self.writes += 1

reconciler = Reconciler()
for desired in [576, 1500, case["mtu"], 9216]:
    reconciler.ensure(desired)
    writes = reconciler.writes
    reconciler.ensure(desired)
    assert reconciler.writes == writes
    assert reconciler.value == desired
print("idempotency holds for normal and boundary inputs")


idempotency holds for normal and boundary inputs


## Check your understanding: Testing fakes, mocks, and invariants

**You should now be able to:** Verify outcomes and required interactions using fake transports, autospec, injected clocks, and negative tests.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.